# BTIS3043 Final Assessment — Intelligent eBook Query System

This notebook is the executable evidence for the two fixed scenarios. The workflow follows the required design:

**Scenario request → dataset-specific predicate query → predicate-only results → fuzzy evaluation → fuzzy-enhanced ranking → comparison and analysis**

The three datasets are **not merged** because their fields and available evidence differ.

In [1]:
import pandas as pd
from IPython.display import display

from src.data_loader import load_datasets, dataset_profile, DATASET_SPECS
from src.knowledge_base import get_scenario
from src.predicate_engine import basic_predicate_query, combined_predicate_query
from src.fuzzy_engine import apply_fuzzy_evaluation, explain_record
from src.evaluation import (
    run_scenario,
    comparison_summary,
    top_results,
    sensitivity_summary,
)

pd.set_option("display.max_colwidth", 90)
pd.set_option("display.width", 180)

## 1. Dataset and knowledge representation

Each dataset keeps its own search fields. This preserves the available knowledge instead of forcing a common schema. Dataset A is the existing/current collection, Dataset B has four levels of discipline detail but no comparable price, and Dataset C provides category/discipline plus licensing prices.

In [2]:
datasets = load_datasets(data_dir="data")
profile = dataset_profile(datasets)
display(profile)

,Dataset,Role,Records,Search fields,Discipline detail,Year evidence,Format evidence,Comparable price,Selected price field
0,A,Current/existing collection,9,Title,None; title only,Yes,No,Yes,Unit Net Price
1,B,Academic/vendor catalogue,1743,"Title, Discipline (Level 1), Discipline (Level 2), Discipline (Level 3), Discipline (L...",Four discipline levels,Yes,Yes,No,Not available
2,C,Potential acquisition catalogue,807,"Title, Category, Discipline",Category + discipline,Yes,Yes,Yes,Single user / 1-Year


## 2. Predicate design: basic and combined predicates

The system explicitly demonstrates both types required by the assessment:

- **Basic predicate:** direct-topic relationship only.
- **Combined predicate:** Boolean combination of direct and supporting relationships.

Scenario 1 uses `Direct_AI OR Programming_Support OR Mathematical_Support`.

Scenario 2 uses `Direct_Security OR Security_Related_Support`; the generic word **security** is accepted only when computing context is present, preventing unrelated matches such as *Food Security*.

In [3]:
for sid in ("S1", "S2"):
    scenario = get_scenario(sid)
    print(f"{sid}: {scenario['name']}")
    print("Combined predicate expression:", scenario["predicate_expression"])
    print()

S1: Artificial Intelligence, Programming and Mathematical Foundations
Combined predicate expression: Direct_AI OR Programming_Support OR Mathematical_Support

S2: Cybersecurity and Secure Computing
Combined predicate expression: Direct_Security OR Security_Related_Support, with a computing-context guard for the generic word 'security'



In [4]:
predicate_demo_rows = []
for sid in ("S1", "S2"):
    for key in ("A", "B", "C"):
        basic = basic_predicate_query(datasets[key], key, sid)
        combined = combined_predicate_query(datasets[key], key, sid)
        predicate_demo_rows.append({
            "Scenario": sid,
            "Dataset": key,
            "Basic direct matches": len(basic),
            "Combined predicate matches": len(combined),
            "Additional candidates from combined logic": len(combined) - len(basic),
        })

display(pd.DataFrame(predicate_demo_rows))

,Scenario,Dataset,Basic direct matches,Combined predicate matches,Additional candidates from combined logic
0,S1,A,0,0,0
1,S1,B,6,234,228
2,S1,C,5,110,105
3,S2,A,1,1,0
4,S2,B,9,9,0
5,S2,C,5,7,2


### Security false-positive check

A generic word such as *security* is not enough by itself. The following small test shows the contextual guard: *Security in Computing* is accepted, while *Food Security* is rejected.

In [5]:
security_test = pd.DataFrame({
    "Title": ["Understanding Food Security", "Security in Computing"],
    "Copyright Year": [2024, 2024],
    "Unit Net Price": [100.0, 100.0],
    "_source_order": [1, 2],
})
security_test_result = combined_predicate_query(security_test, "A", "S2")
display(security_test_result[["Title", "Relationship", "Matched_Terms"]])

,Title,Relationship,Matched_Terms
0,Security in Computing,Direct Security,"security in computing, security"


## 3. Fuzzy reasoning design

Predicate reasoning answers **which records satisfy the scenario**. Fuzzy reasoning then answers **to what degree the accepted records are suitable**.

Four fuzzy preference components are used where evidence is available:

1. **Topic relevance** — direct relationships receive the strongest membership; support relationships are high but lower. A title match is stronger than metadata-only evidence.
2. **Recency** — piecewise-linear membership from recent to older publications, using 2026 as the assessment year.
3. **Format suitability** — ePub/PDF = strongest; Adobe Reader is still treated as a suitable digital format rather than an unknown format.
4. **Affordability** — catalogue-relative membership using Q25 and Q90 of the selected comparable price field.

Default aggregation weights are `relevance=0.45`, `recency=0.25`, `format=0.15`, `affordability=0.15`.

If evidence is unavailable, that component is **excluded** and the remaining weights are re-normalised:

\[
Suitability = \\frac{\sum_{i\in A} w_i\mu_i}{\sum_{i\in A} w_i}
\]

where `A` is the set of available fuzzy components for that record.

In [6]:
for sid in ("S1", "S2"):
    print(sid, get_scenario(sid)["weights"])

S1 {'relevance': 0.45, 'recency': 0.25, 'format': 0.15, 'affordability': 0.15}
S2 {'relevance': 0.45, 'recency': 0.25, 'format': 0.15, 'affordability': 0.15}


# 4. Fixed Scenario 1 — Artificial Intelligence, Programming and Mathematical Foundations

The final candidate set uses the **combined predicate**. Fuzzy evaluation then ranks those candidates. Up to five records are shown per dataset as required.

In [7]:
s1 = run_scenario(datasets, "S1")
s1_summary = comparison_summary(datasets, s1, "S1")
display(s1_summary)

,Scenario,Dataset,Dataset size,Basic direct matches,Combined predicate matches,Match rate %,Top fuzzy score,Top relationship,Search structure,Format evidence,Price evidence,Fuzzy ranking changed order
0,S1,A,9,0,0,0.00,NaN,No match,None; title only,No,Yes,False
1,S1,B,1743,6,234,13.43,1.00,Direct AI,Four discipline levels,Yes,No,True
2,S1,C,807,5,110,13.63,0.95,Direct AI,Category + discipline,Yes,Yes,True


In [8]:
for key in ("A", "B", "C"):
    print()
    print(f"Dataset {key}: {DATASET_SPECS[key]['name']}")
    print(f"Predicate-only matches: {len(s1[key]['predicate'])}")
    if s1[key]["predicate"].empty:
        print("No record satisfied the Scenario 1 predicate in this dataset.")
    else:
        display(top_results(s1[key]["fuzzy"], 5))


Dataset A: Existing eBook Collection
Predicate-only matches: 0
No record satisfied the Scenario 1 predicate in this dataset.

Dataset B: Academic eBook Catalogue
Predicate-only matches: 234


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,1,0,Artificial Intelligence: A Guide to Intelligent Systems,Direct AI,"artificial intelligence, intelligent systems","Title, Discipline (Level 3), Discipline (Level 4)",1.00,1.0,1.0,NaN,1.0000,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
1,2,3,1,"Artificial Intelligence: A Modern Approach, Global Edition\n",Direct AI,artificial intelligence,"Title, Discipline (Level 3), Discipline (Level 4)",1.00,0.8,1.0,NaN,0.9412,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
2,3,35,32,"C++ How to Program, Global Edition",Programming Support,"c++, programming","Title, Discipline (Level 3), Discipline (Level 4)",0.81,1.0,1.0,NaN,0.8994,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
3,4,36,32,"C++ How to Program, Global Edition",Programming Support,"c++, programming","Title, Discipline (Level 3), Discipline (Level 4)",0.81,1.0,1.0,NaN,0.8994,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
4,5,128,123,"Introduction to Java Programming and Data Structures, Global Edition",Programming Support,"java, programming, data structures","Title, Discipline (Level 3), Discipline (Level 4)",0.81,1.0,1.0,NaN,0.8994,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...



Dataset C: eBook Acquisition Catalogue
Predicate-only matches: 110


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,2,1,"Artificial Intelligence, 2e",Direct AI,artificial intelligence,Title,1.00,0.8,1.0,1.0000,0.9500,Main positive driver: topic relevance.
1,2,3,1,China’s Robots,Direct AI,robots,Title,1.00,0.8,1.0,1.0000,0.9500,Main positive driver: topic relevance.
2,3,5,2,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",Direct AI,"artificial intelligence, programming","Title, Discipline",1.00,0.7,1.0,1.0000,0.9250,Main positive driver: topic relevance.
3,4,4,0,Introduction to Artificial Intelligence: A Business Perspective,Direct AI,artificial intelligence,Title,1.00,1.0,1.0,0.4973,0.9246,Main positive driver: topic relevance.
4,5,24,19,Android Boot Camp for Developers Using Java®,Programming Support,"java, programming","Title, Discipline",0.81,1.0,1.0,0.9227,0.9029,Main positive driver: topic relevance.


### Scenario 1 selected decision explanations

The explanation states why a record satisfied the crisp predicate and shows the fuzzy memberships that produced the final rank.

In [9]:
for key in ("B", "C"):
    top = s1[key]["fuzzy"].iloc[0]
    print(f"Dataset {key} top-ranked record:")
    print(top["Title"])
    print(explain_record(top, key))
    print()

Dataset B top-ranked record:
Artificial Intelligence: A Guide to Intelligent Systems
Relationship=Direct AI; matched terms=artificial intelligence, intelligent systems; matched fields=Title, Discipline (Level 3), Discipline (Level 4); relevance=1.00; recency=1.00; format=1.00; affordability=NA; final=1.0000; Main positive driver: topic relevance. Unavailable evidence excluded and weights re-normalised: affordability.

Dataset C top-ranked record:
Artificial Intelligence, 2e
Relationship=Direct AI; matched terms=artificial intelligence; matched fields=Title; relevance=1.00; recency=0.80; format=1.00; affordability=1.00; final=0.9500; price field=Single user / 1-Year; Main positive driver: topic relevance.



# 5. Fixed Scenario 2 — Cybersecurity and Secure Computing

Direct security and related secure-computing references are represented separately. This avoids assigning the same topic-relevance membership to every security-adjacent term. Dataset A is the existing/current collection, so all of its relevant matches are displayed. Up to ten records are shown for the larger catalogues.

In [10]:
s2 = run_scenario(datasets, "S2")
s2_summary = comparison_summary(datasets, s2, "S2")
display(s2_summary)

,Scenario,Dataset,Dataset size,Basic direct matches,Combined predicate matches,Match rate %,Top fuzzy score,Top relationship,Search structure,Format evidence,Price evidence,Fuzzy ranking changed order
0,S2,A,9,1,1,11.11,0.8235,Direct Security,None; title only,No,Yes,False
1,S2,B,1743,9,9,0.52,1.0000,Direct Security,Four discipline levels,Yes,No,True
2,S2,C,807,5,7,0.87,1.0000,Direct Security,Category + discipline,Yes,Yes,True


In [11]:
for key in ("A", "B", "C"):
    print()
    print(f"Dataset {key}: {DATASET_SPECS[key]['name']}")
    print(f"Predicate-only matches: {len(s2[key]['predicate'])}")
    if key == "A":
        print("Dataset A represents the existing/current collection; all relevant current holdings are shown.")
    if s2[key]["predicate"].empty:
        print("No record satisfied the Scenario 2 predicate in this dataset.")
    else:
        display(top_results(s2[key]["fuzzy"], 10))


Dataset A: Existing eBook Collection
Predicate-only matches: 1
Dataset A represents the existing/current collection; all relevant current holdings are shown.


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,1,0,Security in Computing,Direct Security,"security in computing, security",Title,1.0,1.0,NaN,0.0,0.8235,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...



Dataset B: Academic eBook Catalogue
Predicate-only matches: 9


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,2,1,"Computer Security: Principles and Practice, Global Edition",Direct Security,"computer security, security",Title,1.0,1.00,1.00,NaN,1.0000,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
1,2,9,7,Security in Computing,Direct Security,"security in computing, security",Title,1.0,1.00,1.00,NaN,1.0000,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
2,3,4,1,"Cryptography and Network Security: Principles and Practice, Global Edition",Direct Security,"network security, security, computer security, cryptography","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.80,0.85,NaN,0.9147,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
3,4,7,3,"Network Security Essentials: Applications and Standards, Global Edition",Direct Security,"network security, security, computer security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.54,0.85,NaN,0.8382,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
4,5,3,-2,"Computer Security: Principles and Practice, Global Edition",Direct Security,"computer security, security","Title, Discipline (Level 3), Discipline (Level 4)",1.0,0.46,0.85,NaN,0.8147,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
5,6,1,-5,"Boyle: Corporate Computer Security, Global Edition",Direct Security,"computer security, security, network security","Title, Discipline (Level 4)",1.0,0.26,0.85,NaN,0.7559,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
6,7,6,-1,"Business Data Networks and Security, Global Edition",Direct Security,security,Title,1.0,0.26,0.85,NaN,0.7559,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
7,8,5,-3,Introduction to Computer Security,Direct Security,"computer security, security",Title,1.0,0.22,0.85,NaN,0.7441,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...
8,9,8,-1,Practical Cryptology and Web Security,Direct Security,security,Title,1.0,0.10,0.85,NaN,0.7088,Main positive driver: topic relevance. Unavailable evidence excluded and weights re-no...



Dataset C: eBook Acquisition Catalogue
Predicate-only matches: 7


,Fuzzy_Rank,Predicate_Rank,Rank_Change,Title,Relationship,Matched_Terms,Matched_Fields,Relevance_Score,Recency_Score,Format_Score,Affordability_Score,Fuzzy_Score,Decision_Reason
0,1,5,4,Security Awareness: Applying Practical Cybersecurity in Your World,Direct Security,"cybersecurity, security",Title,1.00,1.0,1.0,1.0000,1.0000,Main positive driver: topic relevance.
1,2,3,1,Management of Cybersecurity,Direct Security,"cybersecurity, security",Title,1.00,1.0,1.0,0.7183,0.9577,Main positive driver: topic relevance.
2,3,1,-2,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),Direct Security,"cybersecurity, security",Title,1.00,1.0,1.0,0.3536,0.9030,Main positive driver: topic relevance.
3,4,2,-2,CompTIA Security+ Guide to Network Security Fundamentals,Direct Security,"network security, security",Title,1.00,1.0,1.0,0.3536,0.9030,Main positive driver: topic relevance.
4,5,4,-1,Principles of Information Security,Direct Security,"information security, security",Title,1.00,0.8,1.0,0.3536,0.8530,Main positive driver: topic relevance.
5,6,6,0,Guide to Computer Forensics and Investigations,Security-Related Support,computer forensics,Title,0.78,1.0,1.0,0.3536,0.8040,Main positive driver: topic relevance.
6,7,7,0,Principles of Incident Response & Disaster Recovery,Security-Related Support,incident response,Title,0.78,0.8,1.0,0.3536,0.7540,Main positive driver: topic relevance.


### Selected affordability evidence

Dataset A uses **Unit Net Price**. Dataset B has no comparable price field, so affordability is excluded rather than invented. Dataset C uses **Single user / 1-Year** as the selected licence arrangement.

In [12]:
for sid, outputs in [("S1", s1), ("S2", s2)]:
    print(sid)
    for key in ("A", "B", "C"):
        field = DATASET_SPECS[key]["price_field"]
        thresholds = outputs[key]["affordability_thresholds"]
        print(f"  Dataset {key}: price field={field or 'Not available'}, thresholds={thresholds}")

S1
  Dataset A: price field=Unit Net Price, thresholds=None
  Dataset B: price field=Not available, thresholds=None
  Dataset C: price field=Single user / 1-Year, thresholds={'low': 86.688, 'high': 196.4445}
S2
  Dataset A: price field=Unit Net Price, thresholds={'low': 212.86, 'high': 638.648}
  Dataset B: price field=Not available, thresholds=None
  Dataset C: price field=Single user / 1-Year, thresholds={'low': 86.688, 'high': 196.4445}


### Scenario 2 selected decision explanations

In [13]:
for key in ("A", "B", "C"):
    if not s2[key]["fuzzy"].empty:
        top = s2[key]["fuzzy"].iloc[0]
        print(f"Dataset {key} top-ranked record:")
        print(top["Title"])
        print(explain_record(top, key))
        print()

Dataset A top-ranked record:
Security in Computing
Relationship=Direct Security; matched terms=security in computing, security; matched fields=Title; relevance=1.00; recency=1.00; format=NA; affordability=0.00; final=0.8235; price field=Unit Net Price; Main positive driver: topic relevance. Unavailable evidence excluded and weights re-normalised: format suitability.

Dataset B top-ranked record:
Computer Security: Principles and Practice, Global Edition
Relationship=Direct Security; matched terms=computer security, security; matched fields=Title; relevance=1.00; recency=1.00; format=1.00; affordability=NA; final=1.0000; Main positive driver: topic relevance. Unavailable evidence excluded and weights re-normalised: affordability.

Dataset C top-ranked record:
Security Awareness: Applying Practical Cybersecurity in Your World
Relationship=Direct Security; matched terms=cybersecurity, security; matched fields=Title; relevance=1.00; recency=1.00; format=1.00; affordability=1.00; final=1.00

## 6. Predicate-only vs fuzzy-enhanced comparison

`Predicate_Rank` uses crisp relationship priority and stable catalogue order. `Fuzzy_Rank` uses gradual suitability. `Rank_Change > 0` means the fuzzy stage moved a record upward. This makes the effect of fuzzy reasoning visible rather than only reporting a final score.

In [14]:
rank_change_examples = []
for sid, outputs in [("S1", s1), ("S2", s2)]:
    for key in ("A", "B", "C"):
        f = outputs[key]["fuzzy"]
        if not f.empty:
            moved = f.loc[f["Rank_Change"] != 0, [
                "Title", "Predicate_Rank", "Fuzzy_Rank", "Rank_Change", "Fuzzy_Score"
            ]].head(3).copy()
            moved.insert(0, "Dataset", key)
            moved.insert(0, "Scenario", sid)
            rank_change_examples.append(moved)

if rank_change_examples:
    display(pd.concat(rank_change_examples, ignore_index=True))
else:
    print("No rank changes occurred.")

,Scenario,Dataset,Title,Predicate_Rank,Fuzzy_Rank,Rank_Change,Fuzzy_Score
0,S1,B,"Artificial Intelligence: A Modern Approach, Global Edition\n",3,2,1,0.9412
1,S1,B,"C++ How to Program, Global Edition",35,3,32,0.8994
2,S1,B,"C++ How to Program, Global Edition",36,4,32,0.8994
3,S1,C,"Artificial Intelligence, 2e",2,1,1,0.9500
4,S1,C,China’s Robots,3,2,1,0.9500
5,S1,C,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",5,3,2,0.9250
6,S2,B,"Computer Security: Principles and Practice, Global Edition",2,1,1,1.0000
7,S2,B,Security in Computing,9,2,7,1.0000
8,S2,B,"Cryptography and Network Security: Principles and Practice, Global Edition",4,3,1,0.9147
9,S2,C,Security Awareness: Applying Practical Cybersecurity in Your World,5,1,4,1.0000


## 7. Cross-dataset evaluation

The summary below provides executable evidence for the required discussion of dataset size, topic coverage, attribute structure, discipline detail, price availability and number of matching records.

In [15]:
cross_dataset = pd.concat([s1_summary, s2_summary], ignore_index=True)
display(cross_dataset)

,Scenario,Dataset,Dataset size,Basic direct matches,Combined predicate matches,Match rate %,Top fuzzy score,Top relationship,Search structure,Format evidence,Price evidence,Fuzzy ranking changed order
0,S1,A,9,0,0,0.00,NaN,No match,None; title only,No,Yes,False
1,S1,B,1743,6,234,13.43,1.0000,Direct AI,Four discipline levels,Yes,No,True
2,S1,C,807,5,110,13.63,0.9500,Direct AI,Category + discipline,Yes,Yes,True
3,S2,A,9,1,1,11.11,0.8235,Direct Security,None; title only,No,Yes,False
4,S2,B,1743,9,9,0.52,1.0000,Direct Security,Four discipline levels,Yes,No,True
5,S2,C,807,5,7,0.87,1.0000,Direct Security,Category + discipline,Yes,Yes,True


## 8. Small sensitivity check

Fuzzy weights are a justified design choice, not an objective truth. As a critical check, the code re-ranks Dataset C using a more relevance-heavy alternative (`0.60, 0.20, 0.10, 0.10`). Stable top results suggest robustness; changes show where rankings depend on preference priorities.

In [16]:
alternative_weights = {
    "relevance": 0.60,
    "recency": 0.20,
    "format": 0.10,
    "affordability": 0.10,
}

print("Scenario 1, Dataset C")
display(sensitivity_summary(s1["C"]["fuzzy"], alternative_weights, top_n=5))

print("Scenario 2, Dataset C")
display(sensitivity_summary(s2["C"]["fuzzy"], alternative_weights, top_n=7))

Scenario 1, Dataset C


,_source_order,Title,Fuzzy_Rank,Fuzzy_Score,Alternative_Rank,Alternative_Score,Sensitivity_Rank_Change
21,164,"Artificial Intelligence, 2e",1,0.9500,1,0.96000,0
23,208,China’s Robots,2,0.9500,2,0.96000,0
101,778,"Artificial Intelligence, Analytics and Data Science (Vol. 1)",3,0.9250,4,0.94000,-1
56,422,Introduction to Artificial Intelligence: A Business Perspective,4,0.9246,3,0.94973,1
19,156,Android Boot Camp for Developers Using Java®,5,0.9029,5,0.87827,0


Scenario 2, Dataset C


,_source_order,Title,Fuzzy_Rank,Fuzzy_Score,Alternative_Rank,Alternative_Score,Sensitivity_Rank_Change
6,678,Security Awareness: Applying Practical Cybersecurity in Your World,1,1.0000,1,1.00000,0
3,442,Management of Cybersecurity,2,0.9577,2,0.97183,0
0,229,CompTIA CySA+ Guide to Cybersecurity Analyst (CS0-003),3,0.9030,3,0.93536,0
1,232,CompTIA Security+ Guide to Network Security Fundamentals,4,0.9030,4,0.93536,0
5,668,Principles of Information Security,5,0.8530,5,0.89536,0
2,331,Guide to Computer Forensics and Investigations,6,0.8040,6,0.80336,0
4,667,Principles of Incident Response & Disaster Recovery,7,0.7540,7,0.76336,0


## 9. Implementation strengths, limitations and improvement directions

**Strengths:** dataset-specific predicates, explicit basic/combined logic, direct-vs-support knowledge representation, gradual fuzzy memberships, missing-evidence re-normalisation, explainable ranking, contextual security guard, and reproducible outputs.

**Limitations:** keyword rules do not understand every synonym or semantic meaning; affordability is relative to each catalogue rather than a fixed departmental budget; fuzzy memberships and weights still require expert judgement; metadata-only matches can be broader than title matches.

**Possible improvements:** validate weights with librarian/DCS expert input, add controlled synonyms or ontology terms, use a real departmental budget if supplied, and compare the rule-based method with semantic embeddings in future work. Machine learning is not required for this assessment, so it is intentionally not added to the submitted prototype.